# Code to perform Low Rank Adaptation (LoRA) on the visiion encoders 

In [1]:
import torch
import  torch.nn.functional as F 
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import open_clip
import numpy as np
import pandas as pd
from tqdm import tqdm
import os
from sklearn.metrics import accuracy_score, classification_report, top_k_accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from datetime import datetime
from zoneinfo import ZoneInfo
from typing import Optional, Callable


In [2]:
def evaluate(model,
            text_features : torch.Tensor,
            dl : DataLoader,
            num_classes : int,
            class_offsets,
            loss_func: Optional[Callable] = None,
            CM: Optional[torch.Tensor] = None,
            device : torch.device = torch.device('cpu')
            ):

#        text_features = text_features / torch.linalg.norm(text_features,dim=-1, keepdim=True)

    eval_loss = 0.0
    y_true, y_pred = [], []
    probs_collector = []
    model_dtype = next(model.parameters()).dtype
    running_loss = 0.0
    with torch.no_grad():
        for imgs, labels in dl:
            imgs = imgs.to(device, dtype=model_dtype)
            img_features = model.encode_image(imgs)
            img_features = img_features / torch.linalg.norm(img_features,dim=-1, keepdim=True)

            # native logit scale  * custom temperature
            logit_scale = model.logit_scale.exp() #* temp
            logits = logit_scale * (img_features @ text_features.T)

            # class prompt aggregation
            class_logits = []
            for i in range(num_classes):
                s, e = class_offsets[i], class_offsets[i+1]
                class_logits.append(logits[:, s:e].mean(dim=1))
            class_logits = torch.stack(class_logits, dim=1)

            if loss_func is not None and CM is not None:
                loss = loss_func(class_logits, labels.to(device), CM = CM.to(device) ) ###################### to finish            

            prob = class_logits.softmax(dim=-1)
            pred = prob.argmax(dim=-1)

            y_true.extend(labels.tolist())
            y_pred.extend(pred.tolist())
            probs_collector.append(prob.cpu().numpy())
            if loss_func is not None and CM is not None:
                running_loss += loss.item()
    
    if loss_func is not None and CM is not None:   
        eval_loss = running_loss / len(dl)

    y_true = np.array(y_true, dtype=int)
    y_pred = np.array(y_pred, dtype=int)
    probs = np.concatenate(probs_collector, axis=0)

    top1 = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    top2 = top_k_accuracy_score(y_true, probs, k=min(2, num_classes), labels=np.arange(num_classes))
    top3 = top_k_accuracy_score(y_true, probs, k=min(3, num_classes), labels=np.arange(num_classes))

    return top1, macro_f1, top2, top3, y_true, y_pred, probs, eval_loss


In [3]:
# saves the per class performance in the file ./<CSV_PER_CLASS_FOLDER>/<model_name>.csv
# also saves the confusion matrix in a file ./<CONF_MATRIX_FOLDER>/<model_name
def save_performance_per_class(y_true,y_pred,model_name:str, classnames: list[str], CSV_PER_CLASS_FOLDER, CONF_MATRIX_FOLDER) -> torch.Tensor:
    df=pd.DataFrame(classification_report(y_true, y_pred, target_names=classnames, digits=4 , zero_division=0,output_dict=True))
    df=df.transpose()
    df.columns=["Precision","Recall","F1-score","Support"]
    df.to_csv(os.path.join(CSV_PER_CLASS_FOLDER, model_name+'.csv'), index=True)

    C = torch.from_numpy(confusion_matrix(y_true, y_pred, normalize='true'))
    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay.from_predictions(
        y_true, y_pred,
        display_labels=classnames,
        normalize="true",           # percentuali per riga
        cmap="Blues",
        include_values=False,       # evita numeri sopra le celle se hai tante classi
        xticks_rotation=90,
        ax=ax,
        colorbar=True
    )
    
    ax.set_title("Confusion Matrix (normalized)")
    plt.tight_layout()
    plt.savefig(os.path.join(CONF_MATRIX_FOLDER, model_name))
    plt.close()
    return C

In [4]:
def save_to_csv(row:dict, CSV_PATH):
    df = pd.DataFrame([row])
    display(df)

    # Aggiungi le performance al csv con tutti i modelli
    if os.path.exists(CSV_PATH):
        old = pd.read_csv(CSV_PATH)
        out = pd.concat([old, df], ignore_index=True)
    else:
        out = df.copy()
    out.to_csv(CSV_PATH, index=False)

In [5]:
# From https://github.com/MaxZanella/CLIP-LoRA/blob/main/loralib/layers.py
#  ------------------------------------------------------------------------------------------
#  This code is reconstructed based on loralib (https://github.com/microsoft/LoRA) by Baijiong Lin.
#  ------------------------------------------------------------------------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F

import math
from typing import Optional, List

def set_param(curr_mod, name, param=None, mode='update'):
    r"""Refer to https://github.com/Baijiong-Lin/MOML/blob/main/MTL/utils.py"""
    if '.' in name:
        n = name.split('.')
        module_name = n[0]
        rest = '.'.join(n[1:])
        for name, mod in curr_mod.named_children():
            if module_name == name:
                return set_param(mod, rest, param, mode=mode)
    else:
        if mode == 'update':
            delattr(curr_mod, name)
            setattr(curr_mod, name, param)
        elif mode == 'get':
            if hasattr(curr_mod, name):
                p = getattr(curr_mod, name)
                return p

class LoRALayer():
    def __init__(
        self, 
        r: int, 
        lora_alpha: int, 
        fan_in_fan_out: bool = False,
        dropout_rate:float = 0,
    ):
        self.r = r
        self.lora_alpha = lora_alpha
        self.dropout_rate = dropout_rate
        if self.r > 0:
            #self.scaling = self.lora_alpha / self.r
            self.scaling = self.lora_alpha/math.sqrt(self.r) # 
        # Mark the weight as unmerged
        self.merged = False
        # Set this to True if the layer to replace stores weight like (fan_in, fan_out)
        self.fan_in_fan_out = fan_in_fan_out
        # define params that require LoRA {'param_name': 'lora_name'}
        self.params_with_lora = {}

    def register_lora_param(self):
        r"""Register LoRA matrix"""
        for param_name, lora_name in self.params_with_lora.items():
            assert len(eval(f'self.{param_name}').size()) == 2
            self.register_parameter(f'{lora_name}_lora_A', 
                nn.Parameter(eval(f'self.{param_name}').new_zeros((self.r, eval(f'self.{param_name}').size()[1])))
                )
            self.register_parameter(f'{lora_name}_lora_B', 
                nn.Parameter(eval(f'self.{param_name}').new_zeros((eval(f'self.{param_name}').size()[0], self.r)))
                )
                
            eval(f'self.{param_name}').requires_grad = False

    def init_lora_param(self):
        for param_name, lora_name in self.params_with_lora.items():
            if hasattr(self, f'{lora_name}_lora_A'):
                # initialize A the same way as the default for nn.Linear and B to zero
                nn.init.kaiming_uniform_(eval(f'self.{lora_name}_lora_A'), a=math.sqrt(5))
                nn.init.zeros_(eval(f'self.{lora_name}_lora_B'))

    def transpose(self, w: torch.Tensor):
        return w.transpose(0, 1) if self.fan_in_fan_out else w

    def merge_BA(self, param_name: str):
        lora_name = self.params_with_lora[param_name]
        return self.transpose((eval(f'self.{lora_name}_lora_B') @ eval(f'self.{lora_name}_lora_A')).view(eval(f'self.{param_name}').shape))

    
   
    
    def merge_lora_param(self):
        r"""p_new = p + scaling * B @ A and keep differentiable to A and B"""
        for param_name, lora_name in self.params_with_lora.items():
            p = set_param(self, param_name, mode='get')
            # detach() is very important here
            
            p_new = p.detach() + self.merge_BA(param_name) * self.scaling
            set_param(self, param_name, param=p_new, mode='update')

    def add_lora_data(self):
        r"""NOT differentiable"""
        for param_name, lora_name in self.params_with_lora.items():
            eval(f'self.{param_name}').data += self.merge_BA(param_name) * self.scaling
    
    def sub_lora_data(self):
        r"""NOT differentiable"""
        for param_name, lora_name in self.params_with_lora.items():
            eval(f'self.{param_name}').data -= self.merge_BA(param_name) * self.scaling
            
    
    def lora_train(self, mode: bool = True):
        if mode:
            if self.merged and self.r > 0:
            # Make sure that the weights are not merged
                self.sub_lora_data()
            self.merged = False
        else:
            if not self.merged and self.r > 0:
            # Merge the weights and mark it
                self.add_lora_data()
            self.merged = True 


class LinearLoRA(nn.Linear, LoRALayer):
    # LoRA implemented in a Linear layer
    def __init__(
        self, 
        existing_linear: nn.Linear,
        r: int = 0, 
        lora_alpha: int = 1, 
        fan_in_fan_out: bool = False,
        dropout_rate = 0.,
        **kwargs
    ):
        super().__init__(
            in_features=existing_linear.in_features, 
            out_features=existing_linear.out_features)
        
        self.load_state_dict(existing_linear.state_dict())
        LoRALayer.__init__(self, r=r, lora_alpha=lora_alpha, fan_in_fan_out=fan_in_fan_out)

        # Actual trainable parameters
        self.params_with_lora = {'weight': 'w'}
        if r > 0:
            self.register_lora_param()
        self.init_lora_param()
        self.weight.data = self.transpose(self.weight.data)
        if dropout_rate > 0:
            self.dropout = nn.Dropout(dropout_rate)
        else:
            self.dropout = None

    def train(self, mode: bool = True):
        super().train(mode)     
        self.lora_train(mode)

    def forward(self, x: torch.Tensor, **kwargs):
        # The original implementation had a bug/inefficiency here.
        # This corrected version is much more memory-friendly.

        # Compute the original linear transformation with frozen weights
        original_output = F.linear(x, self.transpose(self.weight), self.bias)

        # Compute and add the LoRA adjustment
        if self.r > 0 and not self.merged:
            
            # Apply dropout to the input if in training mode
            if self.training and self.dropout is not None:
                x_dropout = self.dropout(x)
            else:
                x_dropout = x

            # Get the LoRA matrices. Note: lora_A is already (r, in_features)
            lora_A = self.w_lora_A 
            lora_B = self.w_lora_B
            
            # Efficient computation: (x @ A.T) @ B.T
            # This keeps the intermediate matrix size small (batch_size, r)
            lora_adjustment = (x_dropout @ lora_A.transpose(0, 1)) @ lora_B.transpose(0, 1) * self.scaling
            
            result = original_output + lora_adjustment
        else:
            result = original_output
            
        return result
    # def forward(self, x: torch.Tensor, **kwargs):
        
    #     if self.dropout is None: # do as before
    #         if self.r > 0 and not self.merged:
    #             self.merge_lora_param()
    #             result = nn.Linear.forward(self, x, **kwargs)
    #             self.sub_lora_data()
    #             return result
    #         else:
    #             return nn.Linear.forward(self, x, **kwargs)
            
    #     # Compute the original linear transformation
    #     original_output = nn.Linear.forward(self, x)

    #     if self.training and self.dropout.p > 0:
    #         x = self.dropout(x)
        
    #     if self.r > 0 and not self.merged:
    #         lora_adjustment = torch.matmul(x,self.merge_BA('weight').transpose(0, 1)) * self.scaling 
    #         result = original_output + lora_adjustment
    #     else:
    #         result = original_output
    #     return result


In [6]:
def replace_linear_with_lora(model : torch.nn.Transformer, target_names: list[str], r: int=8, alpha: int=32, lora_dropout: float=0.1):
    for name, mod in model.named_modules():
        if any(target in name for target in target_names) and isinstance(mod, torch.nn.Linear):
            lora = LinearLoRA(mod, r, alpha, dropout_rate=lora_dropout)
            lora.train()
            # Need to use set_param or navigate the hierarchy to replace
            parent_name, child_name = name.rsplit('.', 1)
            parent = model
            for part in parent_name.split('.'):
                parent = getattr(parent, part)
            setattr(parent, child_name, lora)

In [7]:
def loss_weights_func(y : torch.Tensor) -> torch.Tensor:
    a = y.int().bincount()
    w = 1/a
    return w / w.mean()


class weighted_conf_mat_CELoss(torch.nn.Module):
        
    def __init__(self,weights: Optional[torch.Tensor] = None, beta : float =0.05):    
        super().__init__()
        self.W = weights
        self.beta = beta
        self.CE = torch.nn.CrossEntropyLoss(weight=self.W)

    def forward(self,logits: torch.Tensor, labels: torch.Tensor, CM: torch.Tensor ):
        if len(logits.shape) == 1:
            logits = logits.unsqueeze(dim=0)
            labels = labels.unsqueeze(dim=0)
        CM = CM.clone()
        CM[torch.arange(CM.shape[0]), torch.arange(CM.shape[0])] = 0
        logits -= self.beta * CM[labels] 
        return self.CE(logits, labels)

        



In [8]:
model_name = "ViT-L-14"
pretrained = "openai" 
LORA_MODEL_SAVE_PATH = "LoRA"
CSV_PATH = "LoRA_benchmark_results.csv"
CSV_PER_CLASS_FOLDER = "./LoRA_per_class_metrics/"
CONF_MATRIX_FOLDER="./LoRA_confusoin_matrices/"
EPOCHS = 30
BATCH_SIZE = 32
loss_beta = 0.4
TEMPLATES = [
    "a photo of {} architecture",
    "a photograph of {} architecture",
    "an image of {} architecture",
    "a picture of {} architecture",
    "a photo of the {} architectural style",
    "the {} architectural style",
    "a photo of a {} style building",
    "a {} style building",
    "a building in the {} style",
    "{}-style building",
    "a {} building",
    "a building of {} architecture",
    "a representation of {} architecture",
    "the facade of a {} building",
    "an example of {} architecture",
]
if not os.path.isdir(CSV_PER_CLASS_FOLDER):
    os.makedirs(CSV_PER_CLASS_FOLDER)
if not os.path.isdir(CONF_MATRIX_FOLDER):
    os.makedirs(CONF_MATRIX_FOLDER)



In [9]:
model, train_preprocess, test_preprocess = open_clip.create_model_and_transforms(model_name,pretrained)
print(model.visual)


device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")


/home/simone/anaconda3/envs/CV/lib/python3.13/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


VisionTransformer(
  (conv1): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
  (patch_dropout): Identity()
  (ln_pre): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  (transformer): Transformer(
    (resblocks): ModuleList(
      (0-23): 24 x ResidualAttentionBlock(
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=1024, out_features=1024, bias=True)
        )
        (ls_1): Identity()
        (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp): Sequential(
          (c_fc): Linear(in_features=1024, out_features=4096, bias=True)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=4096, out_features=1024, bias=True)
        )
        (ls_2): Identity()
      )
    )
  )
  (ln_post): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
)


In [10]:

train_ds = ImageFolder("../../datasets/architectural-styles-dataset-spl-warp-lanczos/train",transform=train_preprocess) # type:ignore
train_dl = DataLoader(train_ds,batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True )

eval_ds = ImageFolder("../../datasets/architectural-styles-dataset-spl-warp-lanczos/valid",transform=test_preprocess) # type:ignore
eval_dl = DataLoader(eval_ds,batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True )
classnames = train_ds.classes
num_classes = len(train_ds.classes)

### Compute text embeddings for all the prompts 

In [11]:



all_prompts, class_offsets = [], [0]
for cname in classnames:
    ps = [t.format(cname) for t in TEMPLATES] + [f"{cname} architecture"]
    all_prompts.extend(ps)
    class_offsets.append(len(all_prompts))

tokenizer = open_clip.get_tokenizer(model_name)
tokens = tokenizer(all_prompts).to(device)
model.to(device)
with torch.no_grad():
    text_features = model.encode_text(tokens)  # type: ignore
    text_features = text_features / torch.linalg.norm(text_features,dim=-1, keepdim=True)


model.to(torch.device('cpu'))
torch.cuda.empty_cache()


In [ ]:
from torch import autocast, GradScaler

# 1. Freeze everything
for name, p in model.named_parameters():
    p.requires_grad = False

# 2. Inject LoRA (wrap selected Linear layers)
replace_linear_with_lora(model.visual.transformer, target_names=["out_proj", "c_fc", "c_proj"], r=4, alpha=8, lora_dropout=0.35) # type:ignore

# 3. Verify LoRA params are trainable
# for n, p in model.named_parameters():
#     if p.requires_grad:
#         print("Trainable:", n, p.shape)  # should print only lora_A/lora_B (and maybe logit_scale if you unfreeze it)

# --- Model & setup ---

text_features = text_features.to(device)
model.visual = model.visual.to(device)
# Wrap the whole model for multi-GPU (before optimizer creation)
#model.visual = torch.nn.DataParallel(model.visual)

torch.cuda.synchronize()

loss_func = weighted_conf_mat_CELoss(
    weights=loss_weights_func(torch.Tensor(train_ds.targets)).to(device),
    beta=loss_beta
)

optimizer = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=0.0005
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS-10
)

scaler = GradScaler()   # <-- For AMP
model_dtype = next(model.parameters()).dtype
model.train()

train_loss = 0.0
pbar = tqdm(range(EPOCHS))
CM = torch.zeros((num_classes, num_classes))
logit_scale = model.logit_scale.exp()
max_top1 = 0
# --- Training loop ---
for e in pbar:
    # ----- Evaluation -----
    model.eval()
    top1, macro_f1, top2, top3, eval_y_true, eval_y_pred, probs, eval_loss = evaluate(
        model, text_features, eval_dl, num_classes, class_offsets, loss_func, CM, device
    )

    row = {
        "Timestamp": datetime.now(ZoneInfo("Europe/Rome")).isoformat(),
        "Model": model_name,
        "Pretrained": pretrained,
        "Scenario": "LoRA",
        "loss beta": loss_beta,
        " epoch": f"{e}/{EPOCHS}",
        "train loss": train_loss,
        "eval loss": eval_loss,
        "Top-1 Acc": round(float(top1), 4),
        "Top-2 Acc": round(float(top2), 4),
        "Top-3 Acc": round(float(top3), 4),
        "Macro-F1": round(float(macro_f1), 4),
        "Batch size": int(BATCH_SIZE)
    }
    save_to_csv(row, CSV_PATH)
    CM = save_performance_per_class(
        eval_y_true, eval_y_pred, model_name, classnames,
        CSV_PER_CLASS_FOLDER, CONF_MATRIX_FOLDER
    )
    if top1 > max_top1:
        torch.save(model.state_dict(), LORA_MODEL_SAVE_PATH+"_best.pt")
        max_top1 = top1
    # ----- Training -----
    model.train()
    CM = CM.to(device)
    running_loss = 0.0
    torch.cuda.empty_cache()
    optimizer.zero_grad()

    global_imgs, global_labels = [], []

    for i, (imgs, labels) in enumerate(train_dl):
        imgs = imgs.to(device, dtype=model_dtype, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        pbar.set_description(
        f"Eval Accuracy: {top1:.4f}  Eval Loss: {eval_loss:.4f}  Train Loss: {train_loss:.4f} . Batch:{i}/{len(train_dl)}"
        )
        # --- AMP forward pass ---
        #with autocast(device_type="cuda", dtype=torch.float16):
        img_features = model.visual(imgs)  # use .module for DataParallel
        img_features = img_features / torch.linalg.norm(img_features, dim=-1, keepdim=True)
        logits = logit_scale * (img_features @ text_features.T)

        # class-level aggregation
        class_logits = []
        for j in range(num_classes):
            s, e = class_offsets[j], class_offsets[j + 1]
            class_logits.append(logits[:, s:e].mean(dim=1))
        class_logits = torch.stack(class_logits, dim=1).to(device)

        loss = loss_func(class_logits, labels, CM=CM)

        # --- Backward (scaled) ---
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        running_loss += loss.item()

    train_loss = running_loss / len(train_dl)
    CM = CM.to("cpu")
    torch.save(model.state_dict(), LORA_MODEL_SAVE_PATH+"_checkpoint.pt")

# --- Final evaluation & save ---
model.eval()
top1, macro_f1, top2, top3, eval_y_true, eval_y_pred, probs, eval_loss = evaluate(
    model, text_features, eval_dl, num_classes, class_offsets, loss_func, CM, device
)
row = {
    "Timestamp": datetime.now(ZoneInfo("Europe/Rome")).isoformat(),
    "Model": model_name,
    "Pretrained": pretrained,
    "Scenario": "LoRA",
    "loss beta": loss_beta,
    " epoch": f"{e}/{EPOCHS}",
    "Top-1 Acc": round(float(top1), 4),
    "Top-2 Acc": round(float(top2), 4),
    "Top-3 Acc": round(float(top3), 4),
    "Macro-F1": round(float(macro_f1), 4),
    "Batch size": int(BATCH_SIZE)
}
save_to_csv(row, CSV_PATH)
CM = save_performance_per_class(
    eval_y_true, eval_y_pred, model_name, classnames,
    CSV_PER_CLASS_FOLDER, CONF_MATRIX_FOLDER
)
pbar.set_description(
    f"Eval Accuracy: {top1:.4f}  Eval Loss: {eval_loss:.4f}  Train Loss: {train_loss:.4f}"
)
if top1 > max_top1:
    torch.save(model.state_dict(), LORA_MODEL_SAVE_PATH+"_best.pt")
    max_top1 = top1
# Save LoRA weights (unwrapped from DataParallel)



  0%|          | 0/30 [00:00<?, ?it/s]

,Timestamp,Model,Pretrained,Scenario,loss beta,epoch,train loss,eval loss,Top-1 Acc,Top-2 Acc,Top-3 Acc,Macro-F1,Batch size
0,2025-10-27T17:25:16.015319+01:00,ViT-L-14,openai,LoRA,0.4,0/30,0.0,1.395736,0.5804,0.7333,0.8112,0.5618,32


Eval Accuracy: 0.5804  Eval Loss: 1.3957  Train Loss: 0.0000 . Batch:226/227:   3%|▎         | 1/30 [04:59<2:24:51, 299.69s/it]

,Timestamp,Model,Pretrained,Scenario,loss beta,epoch,train loss,eval loss,Top-1 Acc,Top-2 Acc,Top-3 Acc,Macro-F1,Batch size
0,2025-10-27T17:30:16.519627+01:00,ViT-L-14,openai,LoRA,0.4,1/30,0.790547,0.615141,0.8112,0.9109,0.9509,0.811,32


Eval Accuracy: 0.8112  Eval Loss: 0.6151  Train Loss: 0.7905 . Batch:226/227:   7%|▋         | 2/30 [10:03<2:21:04, 302.30s/it]

...

,Timestamp,Model,Pretrained,Scenario,loss beta,epoch,train loss,eval loss,Top-1 Acc,Top-2 Acc,Top-3 Acc,Macro-F1,Batch size
0,2025-10-27T19:50:11.081362+01:00,ViT-L-14,openai,LoRA,0.4,28/30,0.036234,0.884929,0.8182,0.9165,0.9586,0.8177,32


Eval Accuracy: 0.8182  Eval Loss: 0.8849  Train Loss: 0.0362 . Batch:226/227:  97%|█████████▋| 29/30 [2:30:17<05:29, 329.72s/it]

,Timestamp,Model,Pretrained,Scenario,loss beta,epoch,train loss,eval loss,Top-1 Acc,Top-2 Acc,Top-3 Acc,Macro-F1,Batch size
0,2025-10-27T19:55:36.900867+01:00,ViT-L-14,openai,LoRA,0.4,29/30,0.04629,0.874994,0.8098,0.9102,0.9474,0.8094,32


Eval Accuracy: 0.8098  Eval Loss: 0.8750  Train Loss: 0.0463 . Batch:226/227: 100%|██████████| 30/30 [2:35:43<00:00, 311.46s/it]


,Timestamp,Model,Pretrained,Scenario,loss beta,epoch,Top-1 Acc,Top-2 Acc,Top-3 Acc,Macro-F1,Batch size
0,2025-10-27T20:01:02.878665+01:00,ViT-L-14,openai,LoRA,0.4,400/30,0.7726,0.8919,0.934,0.7708,32
